### Import needed libraries

In [2]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import LSTM, Dense, GRU,  Masking, Dropout, Input, BatchNormalization, Layer, GlobalAveragePooling1D, Flatten # type: ignore
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, # type: ignore
                                      TensorBoard, LearningRateScheduler, )

os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'


In [3]:
X = []
y = []

data_dir = "pickles/"

for file in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # list of frames, each frame = list of (x,y)
            if points:
                # Flatten each frame into 1D vector
                seq = [np.array(frame, dtype=np.float32).flatten() for frame in points]
                X.append(seq)
                y.append(entry["class_name"])
    else:
        print(f"{file} not found!")

print(f"Loaded {len(X)} sequences")

Loaded 1800 sequences


In [4]:
max_seq_len = max(len(seq) for seq in X)
feature_dim = max(len(frame) for seq in X for frame in seq)  # largest frame vector size

X_padded = []
for seq in X:
    arr = np.zeros((max_seq_len, feature_dim), dtype=np.float32)
    for i, frame in enumerate(seq):
        arr[i, :len(frame)] = frame
    X_padded.append(arr)

X_padded = np.array(X_padded, dtype=np.float32)  # (num_samples, max_seq_len, feature_dim)
print("X_padded shape:", X_padded.shape)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)

print(X_padded.shape, y_onehot.shape)
print(X_padded[0])
print(X_padded[0][0])

X_padded shape: (1800, 20, 468)
(1800, 20, 468) (1800, 6)
[[228.85187 241.72464 231.61032 ...   0.        0.        0.     ]
 [228.74829 241.68411 231.52026 ...   0.        0.        0.     ]
 [229.00572 242.69765 230.83131 ...   0.        0.        0.     ]
 ...
 [227.82431 242.60811 229.67229 ...   0.        0.        0.     ]
 [228.6836  242.5856  230.53719 ...   0.        0.        0.     ]
 [228.40543 243.42773 230.2832  ...   0.        0.        0.     ]]
[228.85187 241.72464 231.61032 243.56361 231.61032 243.56361 234.36876
 245.40257 234.36876 245.40257 238.0467  248.16103 238.0467  248.16103
 244.4831  250.      244.4831  250.      250.      250.      250.
 250.      257.35587 250.      257.35587 250.      262.87277 248.16103
 262.87277 248.16103 267.4702  245.40257 267.4702  245.40257 270.22867
 243.56361 270.22867 243.56361 272.98712 241.72464 228.85187 241.72464
 230.69083 239.88567 230.69083 239.88567 232.5298  238.0467  232.5298
 238.0467  236.20773 236.20773 236.20773 23

In [10]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import ( # type: ignore
    Input, Masking, Bidirectional, LSTM, LayerNormalization, Dense, Dropout, GlobalAveragePooling1D, MultiHeadAttention
)
from tensorflow.keras.models import Model # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau # type: ignore

# --- Data split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot, test_size=0.8, random_state=42,
    shuffle=True, stratify=y_encoded
)

inputs = Input(shape=(20, 468))

x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
x = Bidirectional(LSTM(128, return_sequences=True))(x)

# Multi-head self-attention
attn = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
x = LayerNormalization()(x + attn)  # residual connection + layer norm

x = GlobalAveragePooling1D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.7)(x)
x = Dense(128, activation="relu")(x)
outputs = Dense(y_onehot.shape[1], activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")



Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.3056 - loss: 1.7492 - val_accuracy: 0.7583 - val_loss: 1.0202 - learning_rate: 1.0000e-04
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5167 - loss: 1.2658 - val_accuracy: 0.7611 - val_loss: 0.7697 - learning_rate: 1.0000e-04
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6222 - loss: 1.0126 - val_accuracy: 0.7757 - val_loss: 0.6657 - learning_rate: 1.0000e-04
Epoch 4/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6583 - loss: 0.8708 - val_accuracy: 0.8368 - val_loss: 0.5342 - learning_rate: 1.0000e-04
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.6694 - loss: 0.8300 - val_accuracy: 0.8090 - val_loss: 0.5664 - learning_rate: 1.0000e-04
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7083 - loss: 0.7379 - val_accuracy: 0.8514 - val_loss: 0.4817 - learning_rate: 1.0000e-04
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 

In [6]:
model.save("gesture_model.keras")

In [7]:
# import libraries
with open('label_encoder.pkl', 'wb') as f:
  pickle.dump(le, f)

In [8]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
try:
    from tensorflow.python.client import device_lib
    print(device_lib.list_local_devices())
except Exception:
    pass


TF: 2.20.0
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 552876486190028863
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5530320896
locality {
  bus_id: 1
  links {
  }
}
incarnation: 18005102150411720980
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]


I0000 00:00:1758536386.425033    9130 gpu_device.cc:2020] Created device /device:GPU:0 with 5274 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


In [9]:
import numpy as np
np.sum(y_train, axis=0)  # class counts


array([60., 60., 60., 60., 60., 60.])